In [1]:
# ============================================================
# 06. CREATE CHROMADB – GESAMTCODE
# ============================================================
#
# Dieser Code:
# 1. erkennt automatisch den OpenLens-Projektordner,
# 2. lädt bereits erzeugte Embeddings,
# 3. lädt die zugehörigen Chunk-Metadaten,
# 4. prüft Anzahl, Dimension und Pflichtfelder,
# 5. erstellt eine persistente lokale ChromaDB,
# 6. speichert Texte, Embeddings und Metadaten,
# 7. prüft die Vollständigkeit,
# 8. führt eine semantische Testsuche durch,
# 9. speichert ein ChromaDB-Manifest.
#
# Voraussetzung:
# Die Embeddings müssen vorher erstellt worden sein.
# ============================================================


# ============================================================
# 1. STANDARD-BIBLIOTHEKEN IMPORTIEREN
# ============================================================

import json
import math
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


# ============================================================
# 2. BENÖTIGTE PAKETE PRÜFEN UND INSTALLIEREN
# ============================================================

def installiere_paket_falls_noetig(
    import_name: str,
    pip_name: str,
) -> None:
    """
    Installiert ein Paket nur, wenn es noch nicht vorhanden ist.
    """

    try:
        __import__(import_name)
        print(f"{pip_name} ist bereits installiert.")

    except ImportError:
        print(f"{pip_name} wird installiert ...")

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--upgrade",
                pip_name,
            ]
        )

        print(f"{pip_name} wurde erfolgreich installiert.")


installiere_paket_falls_noetig(
    import_name="chromadb",
    pip_name="chromadb",
)

installiere_paket_falls_noetig(
    import_name="sentence_transformers",
    pip_name="sentence-transformers",
)

import chromadb
from sentence_transformers import SentenceTransformer


# ============================================================
# 3. EINSTELLUNGEN
# ============================================================

COLLECTION_NAME = "fragdenstaat_openlens"

# Anzahl der Einträge pro ChromaDB-Schreibvorgang.
CHROMA_BATCH_SIZE = 500

# True:
# Vorhandene Collection löschen und sauber neu aufbauen.
#
# False:
# Vorhandene Collection behalten und Einträge aktualisieren.
RESET_COLLECTION = True

# Testfrage für die semantische Suche.
TEST_QUERY = (
    "Dokumente über Informationsfreiheit, "
    "Behördenentscheidungen und staatliche Verwaltung"
)

# Anzahl der angezeigten Suchergebnisse.
TOP_K = 10


# ============================================================
# 4. OPENLENS-PROJEKTORDNER AUTOMATISCH ERKENNEN
# ============================================================

ARBEITSORDNER = Path.cwd().resolve()

BEKANNTE_UNTERORDNER = {
    "datenbank",
    "dokumente",
    "texte",
    "bereinigte_texte",
    "chunks",
    "embeddings",
    "modelle",
    "chromadb",
}

if ARBEITSORDNER.name.lower() in BEKANNTE_UNTERORDNER:
    PROJEKTORDNER = ARBEITSORDNER.parent

elif (ARBEITSORDNER / "Datenbank").is_dir():
    PROJEKTORDNER = ARBEITSORDNER

else:
    raise FileNotFoundError(
        "\nDer OpenLens-Projektordner konnte nicht erkannt werden.\n\n"
        f"Aktueller Arbeitsordner:\n{ARBEITSORDNER}\n\n"
        "Öffne das Notebook entweder aus dem OpenLens-Hauptordner "
        "oder aus dem Ordner OpenLens\\Datenbank."
    )


DATENBANKORDNER = PROJEKTORDNER / "Datenbank"
EMBEDDING_ORDNER = PROJEKTORDNER / "Embeddings"
MODELL_ORDNER = PROJEKTORDNER / "Modelle"
CHROMA_ORDNER = PROJEKTORDNER / "ChromaDB"

CHROMA_ORDNER.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5. DATEIPFADE FESTLEGEN
# ============================================================

EMBEDDINGS_NPY = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embeddings.npy"
)

EMBEDDING_METADATA_JSONL = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embedding_metadata.jsonl"
)

EMBEDDING_METADATA_CSV = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embedding_metadata.csv"
)

EMBEDDING_MANIFEST = (
    EMBEDDING_ORDNER
    / "fragdenstaat_embedding_manifest.json"
)

LOKALES_MODELL = (
    MODELL_ORDNER
    / "paraphrase-multilingual-MiniLM-L12-v2"
)

CHROMA_MANIFEST = (
    CHROMA_ORDNER
    / "openlens_chromadb_manifest.json"
)

TESTERGEBNIS_CSV = (
    CHROMA_ORDNER
    / "chromadb_search_test.csv"
)


print("=" * 80)
print("OPENLENS: CHROMADB AUFBAUEN")
print("=" * 80)

print("\nPython-Umgebung:")
print(sys.executable)

print("\nArbeitsordner:")
print(ARBEITSORDNER)

print("\nProjektordner:")
print(PROJEKTORDNER)

print("\nEmbedding-Ordner:")
print(EMBEDDING_ORDNER)

print("\nChromaDB-Ordner:")
print(CHROMA_ORDNER)

print("\nCollection:")
print(COLLECTION_NAME)


# ============================================================
# 6. EINGABEDATEIEN PRÜFEN
# ============================================================

if not EMBEDDINGS_NPY.exists():
    raise FileNotFoundError(
        "\nDie Embedding-Datei wurde nicht gefunden:\n"
        f"{EMBEDDINGS_NPY}\n\n"
        "Erstelle zuerst die Embeddings."
    )


if not (
    EMBEDDING_METADATA_JSONL.exists()
    or EMBEDDING_METADATA_CSV.exists()
):
    raise FileNotFoundError(
        "\nEs wurden keine Embedding-Metadaten gefunden.\n\n"
        "Erwartet wurde mindestens eine dieser Dateien:\n"
        f"- {EMBEDDING_METADATA_JSONL}\n"
        f"- {EMBEDDING_METADATA_CSV}"
    )


if not EMBEDDING_MANIFEST.exists():
    raise FileNotFoundError(
        "\nDas Embedding-Manifest wurde nicht gefunden:\n"
        f"{EMBEDDING_MANIFEST}"
    )


if not LOKALES_MODELL.is_dir():
    raise FileNotFoundError(
        "\nDas lokale Embedding-Modell wurde nicht gefunden:\n"
        f"{LOKALES_MODELL}\n\n"
        "Das gleiche Modell muss verwendet werden, mit dem "
        "die Embeddings erzeugt wurden."
    )


print("\nAlle benötigten Eingabedateien wurden gefunden.")


# ============================================================
# 7. EMBEDDING-MANIFEST LADEN
# ============================================================

with EMBEDDING_MANIFEST.open(
    "r",
    encoding="utf-8",
) as datei:
    embedding_manifest = json.load(datei)


modell_name = str(
    embedding_manifest.get(
        "model_name",
        "",
    )
)

erwartete_embedding_dimension = int(
    embedding_manifest.get(
        "embedding_dimension",
        -1,
    )
)

erwartete_embedding_anzahl = int(
    embedding_manifest.get(
        "number_of_embeddings",
        -1,
    )
)


print("\nEmbedding-Modell laut Manifest:")
print(modell_name)

print("\nErwartete Embedding-Dimension:")
print(erwartete_embedding_dimension)

print("\nErwartete Embedding-Anzahl:")
print(f"{erwartete_embedding_anzahl:,}")


# ============================================================
# 8. EMBEDDINGS LADEN
# ============================================================

print("\nEmbeddings werden geladen ...")

embeddings = np.load(
    EMBEDDINGS_NPY
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32,
)


if embeddings.ndim != 2:
    raise ValueError(
        "\nDie Embedding-Datei besitzt nicht die erwartete Form.\n"
        f"Gefundene Form: {embeddings.shape}\n"
        "Erwartet wird eine zweidimensionale Matrix."
    )


if not np.isfinite(embeddings).all():
    raise ValueError(
        "\nDie Embeddings enthalten NaN- oder unendliche Werte."
    )


print("\nEmbedding-Matrix:")
print(embeddings.shape)

print("\nEmbedding-Datentyp:")
print(embeddings.dtype)


# ============================================================
# 9. CHUNK-METADATEN LADEN
# ============================================================

if EMBEDDING_METADATA_JSONL.exists():
    print("\nEmbedding-Metadaten werden aus JSONL geladen ...")

    metadata_df = pd.read_json(
        EMBEDDING_METADATA_JSONL,
        lines=True,
    )

    verwendete_metadaten_datei = (
        EMBEDDING_METADATA_JSONL
    )

else:
    print("\nEmbedding-Metadaten werden aus CSV geladen ...")

    metadata_df = pd.read_csv(
        EMBEDDING_METADATA_CSV,
        encoding="utf-8-sig",
        low_memory=False,
    )

    verwendete_metadaten_datei = (
        EMBEDDING_METADATA_CSV
    )


metadata_df = metadata_df.reset_index(
    drop=True
)


print("\nGeladene Metadatensätze:")
print(f"{len(metadata_df):,}")

print("\nVorhandene Spalten:")
print(metadata_df.columns.tolist())


# ============================================================
# 10. PFLICHTSPALTEN PRÜFEN
# ============================================================

PFLICHTSPALTEN = [
    "chunk_id",
    "document_id",
    "text",
]

fehlende_spalten = [
    spalte
    for spalte in PFLICHTSPALTEN
    if spalte not in metadata_df.columns
]


if fehlende_spalten:
    raise KeyError(
        "\nIn den Embedding-Metadaten fehlen Pflichtspalten:\n"
        + "\n".join(
            f"- {spalte}"
            for spalte in fehlende_spalten
        )
    )


print("\nAlle Pflichtspalten sind vorhanden.")


# ============================================================
# 11. ANZAHL UND DIMENSION PRÜFEN
# ============================================================

if len(metadata_df) != embeddings.shape[0]:
    raise ValueError(
        "\nDie Anzahl der Metadaten stimmt nicht mit der "
        "Anzahl der Embeddings überein.\n\n"
        f"Metadaten: {len(metadata_df):,}\n"
        f"Embeddings: {embeddings.shape[0]:,}"
    )


if (
    erwartete_embedding_dimension > 0
    and embeddings.shape[1]
    != erwartete_embedding_dimension
):
    raise ValueError(
        "\nDie Embedding-Dimension stimmt nicht mit dem "
        "Manifest überein.\n\n"
        f"Manifest: {erwartete_embedding_dimension}\n"
        f"Embeddings: {embeddings.shape[1]}"
    )


if (
    erwartete_embedding_anzahl > 0
    and embeddings.shape[0]
    != erwartete_embedding_anzahl
):
    raise ValueError(
        "\nDie Embedding-Anzahl stimmt nicht mit dem "
        "Manifest überein.\n\n"
        f"Manifest: {erwartete_embedding_anzahl:,}\n"
        f"Embeddings: {embeddings.shape[0]:,}"
    )


print("\nAnzahl und Dimension wurden erfolgreich geprüft.")


# ============================================================
# 12. CHUNK-DATEN BEREINIGEN UND PRÜFEN
# ============================================================

metadata_df["chunk_id"] = (
    metadata_df["chunk_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

metadata_df["text"] = (
    metadata_df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

metadata_df["document_id"] = pd.to_numeric(
    metadata_df["document_id"],
    errors="coerce",
)


ungueltige_zeilen = (
    metadata_df["chunk_id"].eq("")
    | metadata_df["text"].eq("")
    | metadata_df["document_id"].isna()
)


if ungueltige_zeilen.any():
    ungueltige_anzahl = int(
        ungueltige_zeilen.sum()
    )

    raise ValueError(
        "\nIn den Embedding-Metadaten befinden sich ungültige "
        "Einträge.\n\n"
        f"Betroffene Zeilen: {ungueltige_anzahl:,}\n\n"
        "Da Embeddings und Metadaten positionsgleich sein müssen, "
        "werden diese Zeilen nicht automatisch entfernt."
    )


metadata_df["document_id"] = (
    metadata_df["document_id"]
    .astype("int64")
)


doppelte_chunk_ids = (
    metadata_df["chunk_id"]
    .duplicated(
        keep=False
    )
)


if doppelte_chunk_ids.any():
    beispiele = (
        metadata_df.loc[
            doppelte_chunk_ids,
            "chunk_id",
        ]
        .head(10)
        .tolist()
    )

    raise ValueError(
        "\nEs wurden doppelte Chunk-IDs gefunden.\n\n"
        f"Beispiele: {beispiele}"
    )


print("\nChunk-Daten wurden erfolgreich geprüft.")


# ============================================================
# 13. METADATEN FÜR CHROMADB VORBEREITEN
# ============================================================

def chroma_metadatenwert(
    wert: Any,
) -> Any:
    """
    Wandelt Werte in ChromaDB-kompatible Datentypen um.

    Erlaubte Typen:
    - String
    - Integer
    - Float
    - Boolean
    """

    if wert is None:
        return ""

    if isinstance(
        wert,
        (
            np.bool_,
            bool,
        ),
    ):
        return bool(wert)

    if isinstance(
        wert,
        (
            np.integer,
            int,
        ),
    ):
        return int(wert)

    if isinstance(
        wert,
        (
            np.floating,
            float,
        ),
    ):
        float_wert = float(wert)

        if math.isnan(float_wert) or math.isinf(float_wert):
            return ""

        return float_wert

    try:
        if pd.isna(wert):
            return ""

    except Exception:
        pass

    return str(wert)


MOEGLICHE_CHROMA_METADATEN_SPALTEN = [
    "document_id",
    "chunk_index",
    "title",
    "pdf_name",
    "page",
    "page_number",
    "page_start",
    "page_end",
    "character_start",
    "character_end",
    "character_count",
    "word_count",
    "site_url",
    "file_url",
    "publicbody",
    "foirequest",
    "uid",
    "source_text_path",
    "clean_text_path",
]


CHROMA_METADATEN_SPALTEN = [
    spalte
    for spalte in MOEGLICHE_CHROMA_METADATEN_SPALTEN
    if spalte in metadata_df.columns
]


def erstelle_chroma_metadaten(
    row: pd.Series,
) -> dict[str, Any]:
    """
    Erstellt ein ChromaDB-kompatibles Metadaten-Dictionary
    für einen einzelnen Chunk.
    """

    metadaten = {}

    for spalte in CHROMA_METADATEN_SPALTEN:
        metadaten[spalte] = chroma_metadatenwert(
            row[spalte]
        )

    return metadaten


print("\nVerwendete ChromaDB-Metadatenspalten:")
print(CHROMA_METADATEN_SPALTEN)

print("\nChromaDB-Metadaten werden vorbereitet ...")

chroma_metadaten = [
    erstelle_chroma_metadaten(row)
    for _, row in metadata_df.iterrows()
]

chunk_ids = (
    metadata_df["chunk_id"]
    .astype(str)
    .tolist()
)

dokumente = (
    metadata_df["text"]
    .astype(str)
    .tolist()
)


print("\nVorbereitete IDs:")
print(f"{len(chunk_ids):,}")

print("\nVorbereitete Dokumenttexte:")
print(f"{len(dokumente):,}")

print("\nVorbereitete Metadatensätze:")
print(f"{len(chroma_metadaten):,}")


# ============================================================
# 14. PERSISTENTEN CHROMADB-CLIENT ERSTELLEN
# ============================================================

print("\nLokaler ChromaDB-Client wird geöffnet ...")

client = chromadb.PersistentClient(
    path=str(CHROMA_ORDNER)
)


# ============================================================
# 15. COLLECTION ZURÜCKSETZEN ODER ÖFFNEN
# ============================================================

vorhandene_collection_namen = [
    vorhandene_collection.name
    for vorhandene_collection in client.list_collections()
]


if (
    RESET_COLLECTION
    and COLLECTION_NAME in vorhandene_collection_namen
):
    print(
        "\nVorhandene Collection wird gelöscht, "
        "damit sie sauber neu aufgebaut werden kann ..."
    )

    client.delete_collection(
        name=COLLECTION_NAME
    )


collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={
        "description": (
            "OpenLens FragDenStaat document chunks"
        ),
        "embedding_model": modell_name,
        "created_by": "OpenLens",
        "distance_metric": "cosine",
    },
    configuration={
        "hnsw": {
            "space": "cosine",
        }
    },
)


print("\nCollection wurde geöffnet:")
print(collection.name)

print("\nBisherige Einträge in der Collection:")
print(f"{collection.count():,}")


# ============================================================
# 16. DATEN STAPELWEISE IN CHROMADB SPEICHERN
# ============================================================

gesamtzahl = len(chunk_ids)

anzahl_batches = math.ceil(
    gesamtzahl
    / CHROMA_BATCH_SIZE
)


print("\n" + "=" * 80)
print("CHUNKS WERDEN IN CHROMADB GESPEICHERT")
print("=" * 80)

print("\nGesamtzahl:")
print(f"{gesamtzahl:,}")

print("\nBatch-Größe:")
print(f"{CHROMA_BATCH_SIZE:,}")

print("\nAnzahl der Batches:")
print(f"{anzahl_batches:,}")


for batch_nummer, start in enumerate(
    range(
        0,
        gesamtzahl,
        CHROMA_BATCH_SIZE,
    ),
    start=1,
):
    ende = min(
        start + CHROMA_BATCH_SIZE,
        gesamtzahl,
    )

    batch_ids = chunk_ids[
        start:ende
    ]

    batch_dokumente = dokumente[
        start:ende
    ]

    batch_metadaten = chroma_metadaten[
        start:ende
    ]

    batch_embeddings = (
        embeddings[
            start:ende
        ]
        .tolist()
    )

    collection.upsert(
        ids=batch_ids,
        embeddings=batch_embeddings,
        documents=batch_dokumente,
        metadatas=batch_metadaten,
    )

    print(
        f"Batch {batch_nummer:,}/{anzahl_batches:,}: "
        f"{ende:,}/{gesamtzahl:,} Einträge gespeichert."
    )


# ============================================================
# 17. VOLLSTÄNDIGKEIT DER DATENBANK PRÜFEN
# ============================================================

gespeicherte_anzahl = collection.count()


print("\n" + "=" * 80)
print("CHROMADB-PRÜFUNG")
print("=" * 80)

print("\nErwartete Datensätze:")
print(f"{gesamtzahl:,}")

print("\nGespeicherte Datensätze:")
print(f"{gespeicherte_anzahl:,}")


if gespeicherte_anzahl != gesamtzahl:
    raise ValueError(
        "\nDie Anzahl der gespeicherten ChromaDB-Einträge "
        "stimmt nicht mit der erwarteten Anzahl überein.\n\n"
        f"Erwartet: {gesamtzahl:,}\n"
        f"Gespeichert: {gespeicherte_anzahl:,}"
    )


print("\nDie ChromaDB enthält alle erwarteten Chunks.")


# ============================================================
# 18. BEISPIELEINTRÄGE ANZEIGEN
# ============================================================

beispiel_anzahl = min(
    5,
    gespeicherte_anzahl,
)

beispiel = collection.peek(
    limit=beispiel_anzahl
)


beispiel_df = pd.DataFrame(
    {
        "chunk_id": beispiel.get(
            "ids",
            [],
        ),
        "text": beispiel.get(
            "documents",
            [],
        ),
        "metadata": beispiel.get(
            "metadatas",
            [],
        ),
    }
)


print("\n" + "=" * 80)
print("BEISPIELEINTRÄGE")
print("=" * 80)

display(beispiel_df)


# ============================================================
# 19. LOKALES EMBEDDING-MODELL FÜR TESTSUCHE LADEN
# ============================================================

print("\nLokales Embedding-Modell wird für den Suchtest geladen ...")

embedding_device = "cpu"

try:
    import torch

    if torch.cuda.is_available():
        embedding_device = "cuda"

except Exception:
    embedding_device = "cpu"


model = SentenceTransformer(
    str(LOKALES_MODELL),
    device=embedding_device,
)


modell_dimension = (
    model.get_sentence_embedding_dimension()
)


if modell_dimension != embeddings.shape[1]:
    raise ValueError(
        "\nDie Dimension des lokalen Modells stimmt nicht "
        "mit der ChromaDB überein.\n\n"
        f"Modell: {modell_dimension}\n"
        f"Datenbank: {embeddings.shape[1]}"
    )


print("\nLokales Embedding-Modell wurde geladen.")

print("\nEmbedding-Gerät:")
print(embedding_device)

print("\nModelldimension:")
print(modell_dimension)


# ============================================================
# 20. SEMANTISCHE TESTSUCHE
# ============================================================

print("\n" + "=" * 80)
print("SEMANTISCHE CHROMADB-TESTSUCHE")
print("=" * 80)

print("\nSuchanfrage:")
print(TEST_QUERY)


query_embedding = model.encode(
    [TEST_QUERY],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

query_embedding = np.asarray(
    query_embedding,
    dtype=np.float32,
)


tatsaechliches_top_k = min(
    TOP_K,
    gespeicherte_anzahl,
)


suchergebnis = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=tatsaechliches_top_k,
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)


ergebnis_ids = (
    suchergebnis.get(
        "ids",
        [[]],
    )[0]
)

ergebnis_dokumente = (
    suchergebnis.get(
        "documents",
        [[]],
    )[0]
)

ergebnis_metadaten = (
    suchergebnis.get(
        "metadatas",
        [[]],
    )[0]
)

ergebnis_distanzen = (
    suchergebnis.get(
        "distances",
        [[]],
    )[0]
)


ergebnis_zeilen = []

for rang, (
    chunk_id,
    dokument,
    metadaten,
    distanz,
) in enumerate(
    zip(
        ergebnis_ids,
        ergebnis_dokumente,
        ergebnis_metadaten,
        ergebnis_distanzen,
    ),
    start=1,
):
    metadaten = metadaten or {}

    # Bei Cosine Distance gilt:
    # kleinere Distanz = größere Ähnlichkeit.
    similarity_score = (
        1.0 - float(distanz)
    )

    ergebnis_zeilen.append(
        {
            "rang": rang,
            "similarity_score": round(
                similarity_score,
                4,
            ),
            "distance": round(
                float(distanz),
                4,
            ),
            "chunk_id": chunk_id,
            "document_id": metadaten.get(
                "document_id",
                "",
            ),
            "title": metadaten.get(
                "title",
                "",
            ),
            "pdf_name": metadaten.get(
                "pdf_name",
                "",
            ),
            "page_start": metadaten.get(
                "page_start",
                "",
            ),
            "page_end": metadaten.get(
                "page_end",
                "",
            ),
            "site_url": metadaten.get(
                "site_url",
                "",
            ),
            "file_url": metadaten.get(
                "file_url",
                "",
            ),
            "text": dokument,
        }
    )


suchergebnis_df = pd.DataFrame(
    ergebnis_zeilen
)


suchergebnis_df.to_csv(
    TESTERGEBNIS_CSV,
    index=False,
    encoding="utf-8-sig",
)


display(suchergebnis_df)


# ============================================================
# 21. CHROMADB-MANIFEST SPEICHERN
# ============================================================

chroma_manifest = {
    "project": "OpenLens",
    "collection_name": COLLECTION_NAME,
    "database_path": str(
        CHROMA_ORDNER.resolve()
    ),
    "distance_metric": "cosine",
    "embedding_model": modell_name,
    "embedding_dimension": int(
        embeddings.shape[1]
    ),
    "number_of_records": int(
        gespeicherte_anzahl
    ),
    "source_embeddings": str(
        EMBEDDINGS_NPY.resolve()
    ),
    "source_metadata": str(
        verwendete_metadaten_datei.resolve()
    ),
    "batch_size": int(
        CHROMA_BATCH_SIZE
    ),
    "reset_collection": bool(
        RESET_COLLECTION
    ),
    "created_at": datetime.now(
        timezone.utc
    ).isoformat(),
}


with CHROMA_MANIFEST.open(
    "w",
    encoding="utf-8",
) as datei:
    json.dump(
        chroma_manifest,
        datei,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# 22. ABSCHLUSSBERICHT
# ============================================================

print("\n" + "=" * 80)
print("CHROMADB ERFOLGREICH AUFGEBAUT")
print("=" * 80)

print("\nCollection:")
print(COLLECTION_NAME)

print("\nGespeicherte Chunks:")
print(f"{gespeicherte_anzahl:,}")

print("\nEmbedding-Dimension:")
print(f"{embeddings.shape[1]:,}")

print("\nDistanzmetrik:")
print("Cosine Distance")

print("\nLokale ChromaDB:")
print(CHROMA_ORDNER)

print("\nChromaDB-Manifest:")
print(CHROMA_MANIFEST)

print("\nTest-Suchergebnisse:")
print(TESTERGEBNIS_CSV)

print(
    "\nDie Collection steht weiterhin in der Variable "
    "'collection' zur Verfügung."
)

print(
    "Die Testtreffer stehen in der Variable "
    "'suchergebnis_df' zur Verfügung."
)

chromadb ist bereits installiert.
sentence-transformers ist bereits installiert.
OPENLENS: CHROMADB AUFBAUEN

Python-Umgebung:
c:\Users\Admin\Desktop\NEUE FISCHE\PANDAS-NUMPY-main\.venv\Scripts\python.exe

Arbeitsordner:
C:\Users\Admin\Desktop\OpenLens\Datenbank

Projektordner:
C:\Users\Admin\Desktop\OpenLens

Embedding-Ordner:
C:\Users\Admin\Desktop\OpenLens\Embeddings

ChromaDB-Ordner:
C:\Users\Admin\Desktop\OpenLens\ChromaDB

Collection:
fragdenstaat_openlens

Alle benötigten Eingabedateien wurden gefunden.

Embedding-Modell laut Manifest:
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Erwartete Embedding-Dimension:
384

Erwartete Embedding-Anzahl:
271

Embeddings werden geladen ...

Embedding-Matrix:
(271, 384)

Embedding-Datentyp:
float32

Embedding-Metadaten werden aus JSONL geladen ...

Geladene Metadatensätze:
271

Vorhandene Spalten:
['embedding_index', 'chunk_id', 'document_id', 'chunk_index', 'title', 'pdf_name', 'page_start', 'page_end', 'character_count', 'wo

,chunk_id,text,metadata
0,8421600000,FEHLER: Nordrhein-Westfalen Drucksache 16/1020...,"{'publicbody': '', 'site_url': 'https://fragde..."
1,8487400000,FEHLER: Nordrhein-Westfalen Drucksache 16/1181...,"{'chunk_index': 0, 'title': 'Inobhutnahme und ..."
2,8497200000,FEHLER: Nordrhein-Westfalen Drucksache 16/1181...,{'source_text_path': 'C:\Users\Admin\Desktop\O...
3,8513800000,FEHLER: Nordrhein-Westfalen Drucksache 16/1020...,"{'foirequest': '', 'pdf_name': '85138_04ce806c..."
4,8597600000,FEHLER: Nordrhein-Westfalen Drucksache 16/1134...,{'uid': '5cec9341-7ddc-4629-896a-fd446328b4f4'...



Lokales Embedding-Modell wird für den Suchtest geladen ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\Admin\AppData\Local\Temp\ipykernel_40868\661799843.py:900: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()



Lokales Embedding-Modell wurde geladen.

Embedding-Gerät:
cuda

Modelldimension:
384

SEMANTISCHE CHROMADB-TESTSUCHE

Suchanfrage:
Dokumente über Informationsfreiheit, Behördenentscheidungen und staatliche Verwaltung


,rang,similarity_score,distance,chunk_id,document_id,title,pdf_name,page_start,page_end,site_url,file_url,text
0,1,0.6144,0.3856,16566300000,165663,Vorbeugegewahrsam,165663_b2e976af7c6eee6bd3676f10c8409a0ec7b7035...,1,2,https://fragdenstaat.de/dokumente/165663-vorbe...,https://media.frag-den-staat.de/files/docs/62/...,Landtag Brandenburg\nDrucksache 3/1493\n3. Wah...
1,2,0.5501,0.4499,16592400000,165924,Private Sicherungsdienste,165924_f917f0b17e9fa7243fe8040c0174905561615c8...,1,2,https://fragdenstaat.de/dokumente/165924-priva...,https://media.frag-den-staat.de/files/docs/71/...,Landtag Brandenburg\nDrucksache 3/3744\n3. Wah...
2,3,0.5186,0.4814,16434800000,164348,Elektronische Fußfessel als Haftersatz,164348_6ed2f459481d69f741e3872e3d702d53d10a866...,1,2,https://fragdenstaat.de/dokumente/164348-elekt...,https://media.frag-den-staat.de/files/docs/54/...,Landtag Brandenburg\nDrucksache 3/138\n3. Wahl...
3,4,0.5107,0.4893,16894100000,168941,Transparenzrichtlinie der Europäischen Union,168941_a5afd00b34b4a932ab13a9b9c23032a07d5d1c9...,1,2,https://fragdenstaat.de/dokumente/168941-trans...,https://media.frag-den-staat.de/files/docs/f1/...,Landtag Brandenburg\nDrucksache 3/1852\n3. Wah...
4,5,0.5063,0.4937,17124200000,171242,Belastung von Verwaltungsgerichten in Asylsachen,171242_ee3aa53faa0625cd0f1d8994504f8a8c8bf6dbc...,1,2,https://fragdenstaat.de/dokumente/171242-belas...,https://media.frag-den-staat.de/files/docs/a1/...,Landtag Brandenburg\nDrucksache 3/1323\n3. Wah...
5,6,0.5046,0.4954,16751500000,167515,Kita-Kürzungen als Grundgesetzauftrag,167515_e96e1be508faab331e34392bf912d388c9d3fad...,1,2,https://fragdenstaat.de/dokumente/167515-kita-...,https://media.frag-den-staat.de/files/docs/4c/...,Landtag Brandenburg\nDrucksache 3/922\n3. Wahl...
6,7,0.4995,0.5005,21947500000,219475,Behandlung einer Landtagseingabe (3318/3/XI),219475_f3a44ce1761fd4841076d542e1423e9087fafc0...,1,1,https://fragdenstaat.de/dokumente/219475-behan...,https://media.frag-den-staat.de/files/docs/6a/...,Niedersächsicher Landtag – 11. Wahlperiode\nDr...
7,8,0.4972,0.5028,17274600000,172746,Begnadigungsrecht,172746_52ecd9e33edaf344b032e94d7b850412f1cd4a5...,1,1,https://fragdenstaat.de/dokumente/172746-begna...,https://media.frag-den-staat.de/files/docs/c4/...,Landtag Brandenburg\nDrucksache 3/1251\n3. Wah...
8,9,0.4883,0.5117,16476800000,164768,"""Schwarze Liste"" des Ministeriums der Justiz u...",164768_721fd9c56259f8103b01ab3346e90e8a79e275d...,1,1,https://fragdenstaat.de/dokumente/164768-schwa...,https://media.frag-den-staat.de/files/docs/96/...,Landtag Brandenburg\nDrucksache 3/2352\n3. Wah...
9,10,0.4855,0.5145,17132200000,171322,Anregungen des Landesrechnungshofs zur Anpassu...,171322_d08d3d24adc21652aded550588fc68aa5133e9d...,1,1,https://fragdenstaat.de/dokumente/171322-anreg...,https://media.frag-den-staat.de/files/docs/d8/...,Landtag Brandenburg\nDrucksache 3/499\n3. Wahl...



CHROMADB ERFOLGREICH AUFGEBAUT

Collection:
fragdenstaat_openlens

Gespeicherte Chunks:
271

Embedding-Dimension:
384

Distanzmetrik:
Cosine Distance

Lokale ChromaDB:
C:\Users\Admin\Desktop\OpenLens\ChromaDB

ChromaDB-Manifest:
C:\Users\Admin\Desktop\OpenLens\ChromaDB\openlens_chromadb_manifest.json

Test-Suchergebnisse:
C:\Users\Admin\Desktop\OpenLens\ChromaDB\chromadb_search_test.csv

Die Collection steht weiterhin in der Variable 'collection' zur Verfügung.
Die Testtreffer stehen in der Variable 'suchergebnis_df' zur Verfügung.
